In [1]:
!pip install --upgrade --force-reinstall huggingface_hub transformers


import huggingface_hub
print(huggingface_hub.__version__)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

def load_student_model():
    print("Loading ltg/gpt-bert-babylm-base...")
    tokenizer = AutoTokenizer.from_pretrained("ltg/gpt-bert-babylm-base")
    model = AutoModelForCausalLM.from_pretrained("ltg/gpt-bert-babylm-base")
    model.eval()
    return tokenizer, model

def generate_student_response(prompt, tokenizer, model, max_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()


  Using cached huggingface_hub-0.34.3-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-4.55.0-py3-none-any.whl.metadata (39 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached hf_xet-1.1.7-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (703 bytes)
  Using cached numpy-2.3.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached regex-2025.7.34-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached to

0.34.3


In [2]:
import huggingface_hub
print(huggingface_hub.__version__)
# =======================
# 🔧 Install dependencies
# =======================
!pip install parlai --quiet

# ===============================
# 📚 Load the teacher model only
# ===============================

from parlai.core.params import ParlaiParser
from parlai.core.agents import create_agent

def load_teacher_agent(model_file='zoo:blender/blender_90M/model'):
    parser = ParlaiParser(True, True, "Teacher model loader")
    parser.set_params(model_file=model_file)
    opt = parser.parse_args([])
    agent = create_agent(opt, requireModelExists=True)
    return agent

# Load teacher
teacher = load_teacher_agent()


0.34.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 8.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.0/209.0 kB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1

20:54:15 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
20:54:15 | Loading model with `--beam-block-full-context false`
20:54:15 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
20:54:15 | num words = 54944
20:54:16 | DEPRECATED: XLM should only be used for backwards compatibility, as it involves a less-stable layernorm operation.
20:54:16 | Total parameters: 87,508,992 (87,508,992 trainable)
20:54:16 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load student model and tokenizer
def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/sam-tokenizer")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/opt-sam-training-preshuffled")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# Generate student response using Hugging Face Transformers
def generate_student_response(prompt, tokenizer, model, device):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=inputs["input_ids"].shape[1] + 50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Teacher-student chat loop
def chat_teacher_student(teacher_agent, student_tokenizer, student_model, device, prompt, num_turns=5):
    print(f"\n[Start Prompt]: {prompt}\n")
    dialogue = prompt.strip()
    teacher_input = prompt.strip()

    for turn in range(num_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # Teacher's turn
        teacher_obs = {'text': teacher_input, 'episode_done': False}
        teacher_agent.observe(teacher_obs)
        teacher_act = teacher_agent.act()
        teacher_reply = teacher_act.get('text', '[No Response]')
        print(f"[Teacher]: {teacher_reply}")
        dialogue += f"\n[Teacher]: {teacher_reply}"

        # Student's turn
        student_input = dialogue + "\n[Student]:"
        student_reply = generate_student_response(student_input, student_tokenizer, student_model, device)
        print(f"[Student]: {student_reply}")
        dialogue += f"\n[Student]: {student_reply}"

        # Next teacher input
        teacher_input = dialogue + "\n[Teacher]:"

In [5]:
# Load teacher agent (you should define this function)
teacher = load_teacher_agent()

# Load the student tokenizer/model/device
student_tokenizer, student_model, device = load_student_model()

# Set initial prompt
initial_prompt = "Hi! I'm interested in learning about space exploration."

# Run the dialogue loop
chat_teacher_student(teacher, student_tokenizer, student_model, device, initial_prompt, num_turns=6)

20:55:07 | Overriding opt["model_file"] to /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model (previously: /checkpoint/edinan/20200210/baseline_BST_retnref/lr=7.5e-06_attention-dropout=0.0_relu-dropout=0.0/model)
20:55:07 | loading dictionary from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model.dict
20:55:07 | num words = 54944
20:55:08 | Total parameters: 87,508,992 (87,508,992 trainable)
20:55:08 | Loading existing model params from /usr/local/lib/python3.11/dist-packages/data/models/blender/blender_90M/model


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]


[Start Prompt]: Hi! I'm interested in learning about space exploration.


--- Turn 1 ---
[Teacher]: i ' d love to learn more about it . what kind of things do you like to do ?
[Student]: they're not like people. not like people, not like people. they are like people. not like humans, not like humans. you want to get out of this world, you want to know what kind of things are. what kind of things

--- Turn 2 ---
[Teacher]: do you have any hobbies ? i like to read , watch movies , and play video games .
[Student]: i've got to go, what do you do ? do you have any hobbies ? not really. i haven't. what do you do ? you have any hobbies ? i'm not sure. you have some hobbies ?

--- Turn 3 ---
[Teacher]: hi ! how are you doing today ? i just got back from playing video games with my friends .
[Student]: what are you doing tonight ? it's so nice of you to say that you and i are just friends, but you're not friends, it's not my style. you don't have any hobbies. i just got back from playing vide